# 01 Exploration: Manheim Used Car Auction Prices

Dieses Notebook ist der erste EDA-Schritt fuer das Projekt **Hybrid AI Agent for Dynamic Used Car Pricing**.
Ziel ist ein schneller, reproduzierbarer Ueberblick ueber Struktur, Vollstaendigkeit und Auffaelligkeiten des Datensatzes, bevor das Micro-Model fuer die fahrzeugspezifische Baseline gebaut wird.

Projektkontext:
- Zielvariable fuer das Micro-Model: `sellingprice` als realer B2B-Auktions-Transaktionspreis.
- Wichtige Fahrzeugfeatures: `year`, `make`, `model`, `trim`, `body`, `transmission`, `condition`, `odometer`.
- Hinweis: Da der Datensatz aus dem US-Markt stammt, ist `odometer` als Meilenstand zu interpretieren, nicht als Kilometerstand.
- `mmr` ist ein Manheim-Marktwert und sollte spaeter eher als Benchmark/Referenz geprueft werden, nicht ungeprueft als Feature, weil es sonst sehr nah an der Zielvariable liegen kann.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

DATA_PATH = Path("car_prices.csv")
DATA_PATH.exists()

## Daten laden

Die Rohdatei enthaelt mindestens eine fehlerhafte CSV-Zeile mit zu vielen Feldern. Fuer diese erste Exploration wird sie uebersprungen, damit die Analyse reproduzierbar durchlaeuft. Die fehlerhafte Zeile sollte vor dem finalen Training separat dokumentiert oder bereinigt werden.

In [ ]:
try:
    df = pd.read_csv(DATA_PATH)
    skipped_bad_lines = False
except pd.errors.ParserError as exc:
    print("ParserError beim direkten Laden:")
    print(exc)
    print("\nLade erneut mit on_bad_lines='skip' fuer die erste Exploration.")
    df = pd.read_csv(DATA_PATH, on_bad_lines="skip")
    skipped_bad_lines = True

print(f"Shape: {df.shape[0]:,} Zeilen x {df.shape[1]:,} Spalten")
print(f"Fehlerhafte CSV-Zeilen uebersprungen: {skipped_bad_lines}")
df.head()

## Struktur: Shape, Spalten, Datentypen

In [ ]:
print("Spalten:")
print(list(df.columns))

schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
})
schema

## Fehlende Werte pro Spalte

In [ ]:
missing = (
    df.isna()
      .sum()
      .rename("missing_count")
      .to_frame()
      .assign(missing_pct=lambda x: (x["missing_count"] / len(df) * 100).round(2))
      .sort_values("missing_count", ascending=False)
)
missing

In [ ]:
plt.figure(figsize=(10, 5))
missing_nonzero = missing[missing["missing_count"] > 0].sort_values("missing_count")
plt.barh(missing_nonzero.index, missing_nonzero["missing_count"], color="#4C78A8")
plt.title("Fehlende Werte pro Spalte")
plt.xlabel("Anzahl fehlender Werte")
plt.ylabel("Spalte")
plt.tight_layout()
plt.show()

## Preis- und Odometer-Verteilung

Die Verteilungen werden zusaetzlich bis zum 99%-Quantil begrenzt geplottet, weil einzelne Extremwerte die Lesbarkeit der Hauptverteilung stark verzerren koennen.

In [ ]:
price_col = "sellingprice"
odometer_col = "odometer"

df[[price_col, odometer_col]].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

price_p99 = df[price_col].quantile(0.99)
odo_p99 = df[odometer_col].quantile(0.99)

sns.histplot(df.loc[df[price_col] <= price_p99, price_col], bins=60, kde=True, ax=axes[0], color="#4C78A8")
axes[0].set_title("Selling Price bis 99%-Quantil")
axes[0].set_xlabel("Selling price in USD")
axes[0].set_ylabel("Anzahl")

sns.histplot(df.loc[df[odometer_col] <= odo_p99, odometer_col], bins=60, kde=True, ax=axes[1], color="#F58518")
axes[1].set_title("Odometer bis 99%-Quantil")
axes[1].set_xlabel("Odometer in miles")
axes[1].set_ylabel("Anzahl")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.boxplot(x=df[price_col], ax=axes[0], color="#4C78A8")
axes[0].set_title("Selling Price: Ausreisser-Check")
axes[0].set_xlabel("Selling price in USD")

sns.boxplot(x=df[odometer_col], ax=axes[1], color="#F58518")
axes[1].set_title("Odometer: Ausreisser-Check")
axes[1].set_xlabel("Odometer in miles")

plt.tight_layout()
plt.show()

## Erste Plausibilitaetschecks

In [ ]:
anomaly_summary = pd.Series({
    "sellingprice <= 0": (df[price_col] <= 0).sum(),
    "sellingprice < 500": (df[price_col] < 500).sum(),
    "sellingprice > 100000": (df[price_col] > 100000).sum(),
    "odometer is missing": df[odometer_col].isna().sum(),
    "odometer <= 0": (df[odometer_col] <= 0).sum(),
    "odometer > 300000": (df[odometer_col] > 300000).sum(),
    "condition is missing": df["condition"].isna().sum(),
    "condition outside 1-5": (~df["condition"].between(1, 5) & df["condition"].notna()).sum(),
})
anomaly_summary.to_frame("count")

In [ ]:
# Beispiele fuer sehr niedrige und sehr hohe Verkaufspreise
cols = ["year", "make", "model", "trim", "body", "condition", "odometer", "mmr", "sellingprice", "saledate"]

low_prices = df.sort_values(price_col).loc[:, cols].head(10)
high_prices = df.sort_values(price_col, ascending=False).loc[:, cols].head(10)

print("Niedrigste Preise:")
display(low_prices)

print("Hoechste Preise:")
display(high_prices)

## Kategorien und Konsistenz

Fuer das spaetere Modell sind Schreibweisen wichtig. Bereits in den ersten Counts sieht man, dass Kategorien wie `Sedan` und `sedan` getrennt auftreten. Das sollte vor Feature Engineering normalisiert werden.

In [ ]:
for col in ["make", "body", "transmission", "state"]:
    print(f"\nTop values fuer {col}:")
    display(df[col].value_counts(dropna=False).head(15).to_frame("count"))

## Notizen: Auffaelligkeiten und naechste Schritte

Erste Auffaelligkeiten aus dieser Exploration:
- Die CSV enthaelt mindestens eine fehlerhafte Zeile mit zu vielen Feldern; fuer die EDA wird sie uebersprungen.
- `transmission`, `body`, `condition`, `trim`, `model` und `make` haben relevante Missing Values und brauchen eine klare Imputations- oder Filterstrategie.
- `sellingprice` ist stark rechtsschief; sehr hohe Preise bis ueber 100.000 USD sind selten, aber fuer Premiumfahrzeuge plausibel zu pruefen.
- Sehr niedrige Verkaufspreise unter 500 USD koennen echte Salvage-/Sonderfaelle oder Datenfehler sein und sollten vor dem Training markiert werden.
- `odometer` ist ebenfalls rechtsschief; extreme Werte bis nahe 1.000.000 miles sollten geprueft und ggf. winsorisiert/gefiltert werden.
- `body` enthaelt inkonsistente Gross-/Kleinschreibung, z.B. `Sedan` und `sedan`. Das ist wichtig fuer das Stage-3-Seasonality-Modell.
- `mmr` ist inhaltlich wertvoll als Markt-Benchmark, aber als Feature fuer `sellingprice` potenziell problematisch, weil es bereits eine Preisreferenz darstellt.

Naechste Schritte fuer das Projekt:
1. Saubere Cleaning-Funktion fuer Bad Lines, Missing Values und Kategorie-Normalisierung bauen.
2. `saledate` parsen und daraus Verkaufsmonat sowie `vehicle_age` ableiten.
3. Baseline-Split nach Zeit oder Sale-Date pruefen, damit die Modellbewertung naeher an der spaeteren Live-Preislogik liegt.
4. Erste Baseline ohne `mmr` trainieren und `mmr` separat als Benchmark vergleichen.